# RAG talent search - offline walkthrough
Runs entirely offline on the **synthetic** sample (45 fictional resumes) with the offline LSA embedder and the template explainer.
For real encoders and an LLM see `02_colab_real_models.ipynb`.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
from talent_rag.config import SAMPLE_DATA
from talent_rag.data import load_resumes, dataset_statistics
from talent_rag.anonymize import anonymize
from talent_rag.index import TalentIndex
from talent_rag.searcher import TalentSearch
from talent_rag.evaluate import build_benchmark, evaluate_index
from talent_rag.bias import run_bias_audit

resumes = load_resumes(SAMPLE_DATA)
print(dataset_statistics(resumes)["resumes"], "resumes;", "annotated labels:", list(dataset_statistics(resumes)["entities_by_label"]))
print(resumes[0].text[:420])

## 1. Privacy first: redact personal data before anything is embedded or sent to an LLM

In [ ]:
r = next(x for x in resumes if "Date of Birth" in x.text)
print(anonymize(r.text, names=r.get("Name"))[:520])

## 2. Chunk, embed and index (vector store: FAISS)

In [ ]:
index = TalentIndex.build(resumes, embedder="lsa:64", store="faiss", anonymize_text=True)
print(index.config["n_resumes"], "resumes ->", index.config["n_chunks"], "chunks | store:", index.config["store"])
index.chunks[0]

## 3. Natural-language search: dense, BM25 and hybrid

In [ ]:
query = "Find me a Junior Data Analyst who knows SQL and Tableau"
search = TalentSearch(index)
for mode in ("dense", "bm25", "hybrid"):
    hits = search.search(query, k=3, mode=mode, rerank_weight=0.0).hits
    print(f"{mode:7s}", [(h.resume.id, h.resume.title) for h in hits])

## 4. Requirement coverage and re-ranking
The query is parsed into skills, seniority and role; candidates are checked against them.

In [ ]:
res = search.search(query, k=3, mode="hybrid", rerank_weight=0.5, explain=True)
print("requirements:", res.spec.requirements())
for hit, cov in zip(res.hits, res.coverages):
    print(hit.rank, hit.resume.name, hit.resume.title, f"coverage={cov.score:.0%}", "missing:", cov.missing_skills)

## 5. Explanation of the top 3 (offline template; plug in an LLM with `llm=OllamaClient(...)`)

In [ ]:
ex = res.explanation
print(ex.backend, "| grounded quotes:", ex.grounded_ratio, "|", ex.overall)
for c in ex.candidates:
    print(c.candidate_id, c.fit_score, c.summary, "| quote:", c.evidence_quotes)

## 6. Evaluate retrieval on an annotation-derived benchmark

In [ ]:
bench = build_benchmark(resumes, index.vocab, n_queries=30)
rows = []
for mode in ("dense", "bm25", "hybrid"):
    means, _ = evaluate_index(index, bench, mode, 0.0)
    rows.append({"mode": mode, **{k: round(v, 3) for k, v in means.items()}})
pd.DataFrame(rows)

## 7. Bias audit: does swapping gendered names / pronouns change the search?

In [ ]:
report = run_bias_audit(resumes, bench[:20], embedder="lsa:64", progress=lambda *_: None)
for r_ in report.counterfactual:
    print("anonymisation", "ON " if r_.anonymized else "OFF", "| top-3 changed for", f"{r_.topk_changed_queries:.0%}", "of queries | max score change", round(r_.max_abs_score_delta, 4))
print(report.flags)

## 8. Use it as a LangChain retriever (optional: `pip install langchain-core`)

In [ ]:
try:
    from talent_rag.langchain_adapter import TalentRetriever
    docs = TalentRetriever(index=index, k=2).invoke("Machine learning engineer with PyTorch and NLP")
    print([(d.metadata["resume_id"], d.metadata["role"]) for d in docs])
except ImportError as exc:
    print(exc)